# KidLisp Canonical Attribution Review

Notebook-first review flow for anonymous-to-canonical KidLisp attribution candidates.

Scope for this phase:
1. Load candidate data + market-style stats
2. Filter/sort candidates for manual review
3. Save decisions and generate a dry-run patch plan JSON

This notebook does **not** apply any live patches.


In [ ]:
from __future__ import annotations

import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List

from IPython.display import HTML, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'reports').exists() and (candidate / 'notebook').exists():
            return candidate
    raise RuntimeError('Could not find repo root from current working directory')


ROOT = find_repo_root()
INPUT_PATH = ROOT / 'reports' / '2026-03-12-jeffrey-kidlisp-attribution-candidates-with-media.json'
DECISIONS_PATH = ROOT / 'reports' / '2026-03-12-kidlisp-canonical-decisions.json'
PATCH_PLAN_PATH = ROOT / 'reports' / '2026-03-12-kidlisp-canonical-patch-plan.json'


def load_dataset(path: Path = INPUT_PATH) -> dict:
    payload = json.loads(path.read_text())
    rows = payload.get('candidates', [])
    for row in rows:
        row['_pair_id'] = f"{row.get('anonCode','?')}->{row.get('matchedJeffCode','?')}"
        row['_confidence_group'] = str(row.get('confidence', '')).split('_')[0]
    payload['candidates'] = rows
    return payload


data = load_dataset()
rows = data['candidates']
ROW_BY_ID = {r['_pair_id']: r for r in rows}

print('repo_root      :', ROOT)
print('input          :', INPUT_PATH)
print('decisions      :', DECISIONS_PATH)
print('patch_plan     :', PATCH_PLAN_PATH)
print('generated_at   :', data.get('generatedAt'))
print('candidate_count:', len(rows))


In [ ]:
def dataset_summary(rows: List[dict]) -> dict:
    confidence = Counter(r.get('confidence', 'unknown') for r in rows)
    total_anon_hits = sum(int(r.get('anonHits') or 0) for r in rows)
    total_match_hits = sum(int(r.get('matchedJeffHits') or 0) for r in rows)
    time_close_1h = sum(1 for r in rows if (r.get('timeDiffHours') is not None and float(r['timeDiffHours']) <= 1))
    exact_like = sum(1 for r in rows if str(r.get('confidence', '')).startswith('high_exact'))
    return {
        'confidence': dict(confidence),
        'total_anon_hits': total_anon_hits,
        'total_match_hits': total_match_hits,
        'time_close_1h': time_close_1h,
        'exact_like': exact_like,
    }


summary = dataset_summary(rows)
summary


In [ ]:
def select_candidates(
    rows: List[dict],
    mode: str = 'all',
    min_hits: int = 0,
    max_time_diff_hours: float | None = None,
    sort_by: str = 'hits_desc',
    limit: int | None = 50,
) -> List[dict]:
    mode = mode.strip().lower()

    def mode_ok(r: dict) -> bool:
        c = str(r.get('confidence', ''))
        hits = int(r.get('anonHits') or 0)
        tdh = r.get('timeDiffHours')
        is_time_close = (tdh is not None and float(tdh) <= 1)
        if mode == 'all':
            return True
        if mode == 'exact':
            return c.startswith('high_exact')
        if mode == 'canonical':
            return 'canonical' in c
        if mode == 'high_impact':
            return hits >= 1000
        if mode == 'time_close':
            return is_time_close
        if mode == 'review_priority':
            return (c.startswith('high_exact') or (hits >= 1000 and 'canonical' in c) or is_time_close)
        raise ValueError(f'Unknown mode: {mode}')

    filtered = []
    for r in rows:
        if int(r.get('anonHits') or 0) < int(min_hits):
            continue
        if max_time_diff_hours is not None:
            tdh = r.get('timeDiffHours')
            if tdh is None or float(tdh) > float(max_time_diff_hours):
                continue
        if mode_ok(r):
            filtered.append(r)

    if sort_by == 'hits_desc':
        filtered.sort(key=lambda r: int(r.get('anonHits') or 0), reverse=True)
    elif sort_by == 'time_asc':
        filtered.sort(key=lambda r: float(r.get('timeDiffHours') or 999999.0))
    elif sort_by == 'confidence_then_hits':
        weight = {'high_exact': 0, 'medium_canonical_timeclose': 1, 'medium_canonical': 2}
        filtered.sort(
            key=lambda r: (
                weight.get(str(r.get('confidence')), 9),
                -int(r.get('anonHits') or 0),
            )
        )
    else:
        raise ValueError(f'Unknown sort_by: {sort_by}')

    if limit is not None:
        return filtered[: int(limit)]
    return filtered


# Quick examples:
print('priority rows:', len(select_candidates(rows, mode='review_priority', limit=None)))
print('exact rows   :', len(select_candidates(rows, mode='exact', limit=None)))
print('high impact  :', len(select_candidates(rows, mode='high_impact', limit=None)))


In [ ]:
def _escape(value: object) -> str:
    text = str(value if value is not None else '')
    return (
        text.replace('&', '&amp;')
        .replace('<', '&lt;')
        .replace('>', '&gt;')
        .replace('"', '&quot;')
    )


def render_candidates_table(candidates: List[dict], decisions: Dict[str, dict] | None = None, title: str = ''):
    decisions = decisions or {}

    css = """
    <style>
      table.ac-review { border-collapse: collapse; width: 100%; font-size: 12px; }
      .ac-review th, .ac-review td { border: 1px solid #d8dee4; padding: 6px; vertical-align: top; }
      .ac-review th { background: #f6f8fa; text-align: left; }
      .ac-code { font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 11px; }
      .ac-muted { color: #57606a; }
      .ac-hit { font-weight: 600; }
      img.ac-thumb { width: 96px; height: 96px; object-fit: cover; border-radius: 6px; border: 1px solid #d0d7de; }
      .ac-title { margin: 6px 0 10px; font-weight: 600; }
    </style>
    """

    parts = [css]
    if title:
        parts.append(f"<div class='ac-title'>{_escape(title)}</div>")

    parts.append('<table class="ac-review">')
    parts.append('<thead><tr>')
    headers = ['pair', 'confidence', 'hits', 'time diff (h)', 'anonymous', 'canonical', 'reason', 'decision']
    for h in headers:
        parts.append(f'<th>{_escape(h)}</th>')
    parts.append('</tr></thead><tbody>')

    for r in candidates:
        pid = r['_pair_id']
        d = decisions.get(pid, {})
        decision_text = d.get('decision', '')
        if d.get('note'):
            decision_text += f" | {_escape(d['note'])}"

        parts.append('<tr>')
        parts.append(f"<td class='ac-code'>{_escape(pid)}</td>")
        parts.append(f"<td>{_escape(r.get('confidence'))}</td>")
        parts.append(
            '<td>'
            f"<span class='ac-hit'>{_escape(r.get('anonHits'))}</span>"
            f" <span class='ac-muted'>(match {_escape(r.get('matchedJeffHits'))})</span>"
            '</td>'
        )
        parts.append(f"<td>{_escape(r.get('timeDiffHours'))}</td>")

        parts.append(
            '<td>'
            f"<div class='ac-code'>${_escape(r.get('anonCode'))}</div>"
            f"<img class='ac-thumb' src='{_escape(r.get('anonStill') or r.get('anonThumb') or '')}' alt='anon'/>"
            '</td>'
        )
        parts.append(
            '<td>'
            f"<div class='ac-code'>${_escape(r.get('matchedJeffCode'))}</div>"
            f"<img class='ac-thumb' src='{_escape(r.get('matchStill') or r.get('matchThumb') or '')}' alt='match'/>"
            '</td>'
        )
        parts.append(f"<td>{_escape(r.get('reason'))}</td>")
        parts.append(f"<td>{_escape(decision_text)}</td>")
        parts.append('</tr>')

    parts.append('</tbody></table>')
    display(HTML(''.join(parts)))


preview = select_candidates(rows, mode='review_priority', sort_by='confidence_then_hits', limit=20)
render_candidates_table(preview, title='Review priority (top 20)')


In [ ]:
VALID_DECISIONS = {'approve', 'hold', 'reject'}


def _now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def load_decisions(path: Path = DECISIONS_PATH) -> Dict[str, dict]:
    if not path.exists():
        return {}
    payload = json.loads(path.read_text())
    return payload.get('decisions', {})


def save_decisions(decisions: Dict[str, dict], path: Path = DECISIONS_PATH) -> Path:
    payload = {
        'generatedAt': _now_iso(),
        'source': str(INPUT_PATH.relative_to(ROOT)),
        'count': len(decisions),
        'decisions': decisions,
    }
    path.write_text(json.dumps(payload, indent=2) + '\n')
    return path


def set_decision(
    anon_code: str,
    matched_code: str,
    decision: str,
    note: str = '',
    reviewer: str = '@jeffrey',
):
    decision = decision.strip().lower()
    if decision not in VALID_DECISIONS:
        raise ValueError(f'decision must be one of {sorted(VALID_DECISIONS)}')

    pair_id = f"{anon_code}->{matched_code}"
    if pair_id not in ROW_BY_ID:
        raise KeyError(f'Unknown pair: {pair_id}')

    decisions = load_decisions()
    decisions[pair_id] = {
        'decision': decision,
        'note': note,
        'reviewer': reviewer,
        'decidedAt': _now_iso(),
    }
    save_decisions(decisions)
    return decisions[pair_id]


def clear_decision(anon_code: str, matched_code: str):
    pair_id = f"{anon_code}->{matched_code}"
    decisions = load_decisions()
    removed = decisions.pop(pair_id, None)
    save_decisions(decisions)
    return removed


def bulk_set_decision(candidates: List[dict], decision: str, note: str = '', reviewer: str = '@jeffrey') -> int:
    decision = decision.strip().lower()
    if decision not in VALID_DECISIONS:
        raise ValueError(f'decision must be one of {sorted(VALID_DECISIONS)}')
    decisions = load_decisions()
    count = 0
    for row in candidates:
        decisions[row['_pair_id']] = {
            'decision': decision,
            'note': note,
            'reviewer': reviewer,
            'decidedAt': _now_iso(),
        }
        count += 1
    save_decisions(decisions)
    return count


def decisions_summary() -> dict:
    decisions = load_decisions()
    return {
        'path': str(DECISIONS_PATH.relative_to(ROOT)),
        'total': len(decisions),
        'by_decision': dict(Counter(v.get('decision', 'unknown') for v in decisions.values())),
    }


# Optional seed: auto-approve exact matches only.
# exact_rows = select_candidates(rows, mode='exact', limit=None)
# bulk_set_decision(exact_rows, decision='approve', note='exact source match')

# Example manual decisions:
# set_decision('bop', 'pi5', 'hold', note='high impact canonical; verify lineage')
# set_decision('ceo', 'puf', 'hold', note='high impact canonical; verify lineage')

decisions = load_decisions()
print(decisions_summary())


In [ ]:
def build_patch_plan(rows: List[dict], decisions: Dict[str, dict]) -> List[dict]:
    plan = []
    for row in rows:
        pair_id = row['_pair_id']
        d = decisions.get(pair_id)
        if not d or d.get('decision') != 'approve':
            continue

        plan.append(
            {
                'pair_id': pair_id,
                'from_code': row.get('anonCode'),
                'to_code': row.get('matchedJeffCode'),
                'confidence': row.get('confidence'),
                'anon_hits': row.get('anonHits'),
                'matched_hits': row.get('matchedJeffHits'),
                'time_diff_hours': row.get('timeDiffHours'),
                'reason': row.get('reason'),
                'source_preview': row.get('sourcePreview'),
                'evidence': {
                    'anon_when': row.get('anonWhen'),
                    'matched_when': row.get('matchedJeffWhen'),
                    'anon_thumb': row.get('anonThumb'),
                    'anon_still': row.get('anonStill'),
                    'match_thumb': row.get('matchThumb'),
                    'match_still': row.get('matchStill'),
                },
                'decision': d,
            }
        )
    return plan


def write_patch_plan(path: Path = PATCH_PLAN_PATH):
    decisions = load_decisions()
    plan_rows = build_patch_plan(rows, decisions)

    payload = {
        'generatedAt': _now_iso(),
        'mode': 'dry-run',
        'source': str(INPUT_PATH.relative_to(ROOT)),
        'decisionsSource': str(DECISIONS_PATH.relative_to(ROOT)),
        'approvedCount': len(plan_rows),
        'entries': plan_rows,
    }

    path.write_text(json.dumps(payload, indent=2) + '\n')
    return {
        'patchPlanPath': str(path.relative_to(ROOT)),
        'approvedCount': len(plan_rows),
        'totalDecisions': len(decisions),
    }


write_patch_plan()


## Suggested Usage

1. Preview priority rows:
   - `preview = select_candidates(rows, mode='review_priority', sort_by='confidence_then_hits', limit=20)`
   - `render_candidates_table(preview, decisions=load_decisions())`
2. Make decisions:
   - `set_decision('r53', 'fm3', 'approve', note='exact source match')`
   - `set_decision('bop', 'pi5', 'hold', note='manual lineage check needed')`
3. Emit dry-run patch plan JSON:
   - `write_patch_plan()`

Outputs:
- `reports/2026-03-12-kidlisp-canonical-decisions.json`
- `reports/2026-03-12-kidlisp-canonical-patch-plan.json`
